# WiDS 2026 Wildfire Survival Breakthrough Stack

In [1]:

import os, re, gc, json, math, time, glob, copy, random, hashlib, warnings
from collections import defaultdict

import numpy as np
import pandas as pd

from scipy.special import expit, logit
from scipy.stats import norm, logistic as logistic_dist
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import ExtraTreesClassifier

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HORIZONS = [12, 24, 48, 72]
INTERVALS = [(0, 12), (12, 24), (24, 48), (48, 72)]
REQUIRED_COLS = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
MAX_THREADS = max(1, min(4, os.cpu_count() or 4))
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "/kaggle/temp"]

SCORE_LOOKUP = {
    1: 0.96617, 2: 0.96540, 3: 0.96961, 4: 0.97252, 5: 0.97167, 6: 0.97204,
    7: 0.97252, 8: 0.97058, 9: 0.95804, 10: 0.97118, 11: 0.96933, 12: 0.96539,
    13: 0.96400, 14: 0.95893, 15: 0.96617, 16: 0.96539, 17: 0.97252, 18: 0.96002,
    19: 0.95628, 20: 0.96216, 21: 0.95357, 22: 0.97252, 23: 0.97191, 24: 0.95820,
    25: 0.95868, 26: 0.96528, 27: 0.97253, 28: 0.94438, 29: 0.96182, 30: 0.95844,
    31: 0.96690, 32: 0.97091,
}

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

print("HAS_XGB:", HAS_XGB, "| HAS_LGB:", HAS_LGB, "| HAS_CAT:", HAS_CAT)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def slog1p(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))


def enforce_monotonicity(arr):
    arr = np.asarray(arr, dtype=float)
    arr = np.clip(arr, 0.0, 1.0)
    return np.maximum.accumulate(arr, axis=1)


def find_core_dataset():
    candidates = []
    for train_path in glob.glob("/kaggle/input/**/train.csv", recursive=True):
        root = os.path.dirname(train_path)
        needed = ["train.csv", "test.csv", "sample_submission.csv"]
        if not all(os.path.exists(os.path.join(root, fn)) for fn in needed):
            continue
        root_low = root.lower()
        score = 0
        if "wids" in root_low:
            score += 10
        if "globaldathon" in root_low or "datathon" in root_low:
            score += 10
        if "wildfire" in root_low:
            score += 5
        if "competition" in root_low:
            score += 2
        score -= len(root_low) / 1000.0
        candidates.append((score, root))
    if not candidates:
        raise FileNotFoundError("Could not find train.csv/test.csv/sample_submission.csv under /kaggle/input")
    candidates = sorted(candidates, reverse=True)
    return candidates[0][1]


def validate_submission(sub, sample_sub):
    if list(sub.columns) != REQUIRED_COLS:
        raise ValueError(f"Submission columns must be exactly {REQUIRED_COLS}")
    if sub["event_id"].duplicated().any():
        raise ValueError("Duplicate event_id found")
    if list(sub["event_id"]) != list(sample_sub["event_id"]):
        raise ValueError("event_id order/content does not match sample submission exactly")
    vals = sub[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if not np.isfinite(vals).all():
        raise ValueError("Non-finite probabilities found")
    if (vals < -1e-12).any() or (vals > 1 + 1e-12).any():
        raise ValueError("Probabilities must be in [0, 1]")
    if (np.diff(vals, axis=1) < -1e-10).any():
        raise ValueError("Row-wise monotonicity violated")
    return True


def _qcut_codes(s, q=4):
    s = pd.Series(s)
    qq = min(q, s.nunique())
    if qq <= 1:
        return pd.Series(np.zeros(len(s), dtype=int), index=s.index)
    try:
        return pd.qcut(s, q=qq, labels=False, duplicates="drop").astype(int)
    except Exception:
        return pd.cut(s, bins=qq, labels=False, include_lowest=True).astype(int)


def make_strat_labels(train_df, min_count=5):
    event = train_df["event"].astype(int).to_numpy()
    time_vals = train_df["time_to_hit_hours"].astype(float).to_numpy()
    out = np.zeros(len(train_df), dtype=int)
    for ev in [0, 1]:
        idx = np.where(event == ev)[0]
        if len(idx) == 0:
            continue
        out[idx] = _qcut_codes(time_vals[idx], q=4).to_numpy()
    if "dist_min_ci_0_5h" in train_df.columns:
        dist_bins = _qcut_codes(train_df["dist_min_ci_0_5h"], q=4).to_numpy()
    else:
        dist_bins = np.zeros(len(train_df), dtype=int)
    labels = pd.Series(event).astype(str) + "_" + pd.Series(out).astype(str) + "_" + pd.Series(dist_bins).astype(str)
    counts = labels.value_counts()
    rare = set(counts[counts < min_count].index)
    if rare:
        labels = labels.where(~labels.isin(rare), pd.Series(event).astype(str) + "_fallback")
    return labels.to_numpy()


def build_cv_splits(train_df, n_splits=5, seeds=(42, 202, 2026)):
    strat_labels = make_strat_labels(train_df, min_count=n_splits)
    splits = []
    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train_df)), strat_labels)):
            splits.append({"seed": seed, "fold": fold, "train_idx": tr_idx, "valid_idx": va_idx})
    return splits


def build_cindex_pairs(time_vals, event_vals):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    left = []
    right = []
    n = len(time_vals)
    for i in range(n):
        ti = time_vals[i]
        ei = event_vals[i]
        for j in range(i + 1, n):
            tj = time_vals[j]
            ej = event_vals[j]
            if ti == tj:
                continue
            if ei == 1 and ti < tj:
                left.append(i)
                right.append(j)
            elif ej == 1 and tj < ti:
                left.append(j)
                right.append(i)
    return np.asarray(left, dtype=int), np.asarray(right, dtype=int)


def cindex_from_pairs(risk, pair_left, pair_right):
    if len(pair_left) == 0:
        return 0.5
    risk = np.asarray(risk, dtype=float)
    rl = risk[pair_left]
    rr = risk[pair_right]
    return (np.sum(rl > rr) + 0.5 * np.sum(rl == rr)) / len(pair_left)


def build_brier_cache(time_vals, event_vals, horizons=HORIZONS):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    cache = {}
    for h in horizons:
        mask = ~((event_vals == 0) & (time_vals < h))
        y = ((event_vals == 1) & (time_vals <= h)).astype(float)
        cache[h] = {"mask": mask, "y": y}
    return cache


def brier_from_cache(pred_col, cache_obj):
    mask = cache_obj["mask"]
    y = cache_obj["y"]
    if mask.sum() == 0:
        return 0.0
    p = np.asarray(pred_col, dtype=float)
    return float(np.mean((p[mask] - y[mask]) ** 2))


def weighted_brier(pred_mat, brier_cache):
    pred_mat = np.asarray(pred_mat, dtype=float)
    return (
        0.3 * brier_from_cache(pred_mat[:, 1], brier_cache[24]) +
        0.4 * brier_from_cache(pred_mat[:, 2], brier_cache[48]) +
        0.3 * brier_from_cache(pred_mat[:, 3], brier_cache[72])
    )


def risk_from_probs(pred_mat):
    pred_mat = np.asarray(pred_mat, dtype=float)
    return 12.0 * pred_mat[:, 0] + 12.0 * pred_mat[:, 1] + 24.0 * pred_mat[:, 2] + 24.0 * pred_mat[:, 3]


def hybrid_score(pred_mat, pair_left, pair_right, brier_cache):
    pred_mat = enforce_monotonicity(pred_mat)
    cidx = cindex_from_pairs(risk_from_probs(pred_mat), pair_left, pair_right)
    wb = weighted_brier(pred_mat, brier_cache)
    return 0.3 * cidx + 0.7 * (1.0 - wb)


def horizon_train_mask(time_vals, event_vals, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    return ~((event_vals == 0) & (time_vals < horizon))


def horizon_target(time_vals, event_vals, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    return ((event_vals == 1) & (time_vals <= horizon)).astype(int)


def class_balance_weights(y, power=0.70, max_pos_weight=6.0):
    y = np.asarray(y, dtype=int)
    pos = int(y.sum())
    neg = int(len(y) - pos)
    if pos == 0 or neg == 0:
        return np.ones(len(y), dtype=float)
    pos_w = min(max_pos_weight, (neg / max(pos, 1)) ** power)
    w = np.ones(len(y), dtype=float)
    w[y == 1] = pos_w
    return w


def aft_cdf(z, dist_name):
    z = np.asarray(z, dtype=float)
    if dist_name == "normal":
        return norm.cdf(z)
    if dist_name == "logistic":
        return logistic_dist.cdf(z)
    if dist_name == "extreme":
        return 1.0 - np.exp(-np.exp(z))
    raise ValueError(f"Unknown AFT distribution: {dist_name}")


def breslow_cumhaz(time_vals, event_vals, hazard_ratio, horizons=HORIZONS):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    hazard_ratio = np.clip(np.asarray(hazard_ratio, dtype=float), 1e-12, None)
    event_times = np.sort(np.unique(time_vals[event_vals == 1]))
    if len(event_times) == 0:
        return {h: 0.0 for h in horizons}
    increments = []
    for t in event_times:
        d = float(np.sum((time_vals == t) & (event_vals == 1)))
        r = float(hazard_ratio[time_vals >= t].sum())
        increments.append(0.0 if r <= 0 else d / r)
    increments = np.asarray(increments, dtype=float)
    cum = np.cumsum(increments)
    out = {}
    for h in horizons:
        idx = np.searchsorted(event_times, h, side="right") - 1
        out[h] = float(cum[idx]) if idx >= 0 else 0.0
    return out


def _series_or_default(df, col, default=0.0):
    if col in df.columns:
        return df[col].astype(float)
    return pd.Series(default, index=df.index, dtype=float)


def engineer_features(df):
    df = df.copy()

    cyc_map = {"event_start_hour": 24, "event_start_dayofweek": 7, "event_start_month": 12}
    for col, period in cyc_map.items():
        if col in df.columns:
            angle = 2.0 * np.pi * df[col].astype(float) / period
            df[f"{col}_sin"] = np.sin(angle)
            df[f"{col}_cos"] = np.cos(angle)

    d = _series_or_default(df, "dist_min_ci_0_5h", 10000.0).clip(lower=1.0)
    closing = _series_or_default(df, "closing_speed_m_per_h", 0.0)
    closing_abs = _series_or_default(df, "closing_speed_abs_m_per_h", np.nan).fillna(np.abs(closing))
    along = _series_or_default(df, "along_track_speed", 0.0)
    radial = _series_or_default(df, "radial_growth_rate_m_per_h", 0.0)
    advance = _series_or_default(df, "projected_advance_m", 0.0)
    align = _series_or_default(df, "alignment_cos", 0.0)
    align_abs = _series_or_default(df, "alignment_abs", np.nan).fillna(np.abs(align))
    cross = _series_or_default(df, "cross_track_component", 0.0)
    area0 = _series_or_default(df, "area_first_ha", 0.0).clip(lower=0.0)
    grow_abs = _series_or_default(df, "area_growth_abs_0_5h", 0.0).clip(lower=0.0)
    grow_rel = _series_or_default(df, "area_growth_rel_0_5h", 0.0)
    grow_rate = _series_or_default(df, "area_growth_rate_ha_per_h", 0.0)
    log_area0 = _series_or_default(df, "log1p_area_first", np.log1p(area0))
    log_grow = _series_or_default(df, "log1p_growth", np.log1p(grow_abs))
    dist_slope_close = -_series_or_default(df, "dist_slope_ci_0_5h", 0.0)
    dist_change_close = -_series_or_default(df, "dist_change_ci_0_5h", 0.0)
    accel_close = -_series_or_default(df, "dist_accel_m_per_h2", 0.0)
    cent_speed = _series_or_default(df, "centroid_speed_m_per_h", 0.0)
    r2 = _series_or_default(df, "dist_fit_r2_0_5h", 0.0).clip(lower=0.0, upper=1.0)
    low_res = _series_or_default(df, "low_temporal_resolution_0_5h", 0.0)
    dt = _series_or_default(df, "dt_first_last_0_5h", 1.0).clip(lower=1e-3)
    nper = _series_or_default(df, "num_perimeters_0_5h", 1.0).clip(lower=1.0)
    area_ratio_log = _series_or_default(df, "log_area_ratio_0_5h", 0.0)
    radial_growth = _series_or_default(df, "radial_growth_m", 0.0)

    dist_km = d / 1000.0
    close_pos = np.maximum(closing, 0.0)
    along_pos = np.maximum(along, 0.0)
    radial_pos = np.maximum(radial, 0.0)
    slope_pos = np.maximum(dist_slope_close, 0.0)
    accel_pos = np.maximum(accel_close, 0.0)

    toward_speed = 0.55 * close_pos + 0.30 * along_pos + 0.15 * radial_pos
    wavefront_speed = radial_pos + 0.70 * align_abs * along_pos + 0.35 * close_pos
    slope_speed = slope_pos + 0.30 * radial_pos

    eta_close = d / np.maximum(close_pos, 1.0)
    eta_along = d / np.maximum(along_pos, 1.0)
    eta_toward = d / np.maximum(toward_speed, 1.0)
    eta_wave = d / np.maximum(wavefront_speed, 1.0)
    eta_slope = d / np.maximum(slope_speed, 1.0)

    eta_stack = np.column_stack([eta_close, eta_along, eta_toward, eta_wave, eta_slope])
    eta_min = np.min(eta_stack, axis=1)
    eta_mean = np.mean(eta_stack, axis=1)
    eta_hmean = eta_stack.shape[1] / np.sum(1.0 / np.maximum(eta_stack, 1e-3), axis=1)

    df["dist_km"] = dist_km
    df["log_dist_m"] = np.log1p(d)
    df["inv_dist_km"] = 1.0 / (1.0 + dist_km)
    df["close_per_km"] = close_pos / (dist_km + 0.25)
    df["along_per_km"] = along_pos / (dist_km + 0.25)
    df["radial_per_km"] = radial_pos / (dist_km + 0.25)
    df["slope_per_km"] = slope_pos / (dist_km + 0.25)
    df["toward_speed_mph"] = toward_speed
    df["wavefront_speed_mph"] = wavefront_speed
    df["slope_speed_mph"] = slope_speed
    df["eta_close_h"] = np.clip(eta_close, 0.0, 500.0)
    df["eta_along_h"] = np.clip(eta_along, 0.0, 500.0)
    df["eta_toward_h"] = np.clip(eta_toward, 0.0, 500.0)
    df["eta_wave_h"] = np.clip(eta_wave, 0.0, 500.0)
    df["eta_slope_h"] = np.clip(eta_slope, 0.0, 500.0)
    df["eta_min_h"] = np.clip(eta_min, 0.0, 500.0)
    df["eta_mean_h"] = np.clip(eta_mean, 0.0, 500.0)
    df["eta_hmean_h"] = np.clip(eta_hmean, 0.0, 500.0)
    df["aligned_close"] = align_abs * close_pos
    df["aligned_along"] = align_abs * along_pos
    df["aligned_wavefront"] = align_abs * wavefront_speed
    df["aligned_slope"] = align_abs * slope_pos
    df["advance_ratio"] = advance / (d + 1.0)
    df["closing_quality"] = slope_pos * (0.50 + 0.50 * r2)
    df["directional_pressure"] = (1.0 + align) * (close_pos + 0.50 * along_pos + 0.20 * radial_pos)
    df["growth_pressure"] = (1.0 + log_area0 + 0.5 * log_grow) * (1.0 + grow_rel.clip(lower=-0.95))
    df["threat_gravity"] = ((1.0 + log_area0 + 0.6 * log_grow) * (1.0 + align_abs) * (1.0 + r2)) / (dist_km + 0.75)
    df["threat_momentum"] = (0.60 * close_pos + 0.25 * along_pos + 0.15 * radial_pos) * (1.0 + align_abs) / (dist_km + 0.50)
    df["motion_pressure"] = (1.0 + np.log1p(np.maximum(cent_speed, 0.0))) * (1.0 + align_abs) / (dist_km + 0.80)
    df["size_pressure"] = (1.0 + log_area0) * (1.0 + np.log1p(np.maximum(grow_rate, 0.0))) / (dist_km + 0.80)
    df["resolution_penalty"] = low_res * dist_km
    df["coverage_density"] = nper / (dt + 0.25)
    df["cross_track_abs"] = np.abs(cross)
    df["radial_vs_centroid"] = radial_pos / (np.maximum(cent_speed, 1.0))
    df["advance_per_hour"] = advance / (dt + 1e-3)
    df["dist_change_per_hour"] = dist_change_close / (dt + 1e-3)
    df["accel_pressure"] = accel_pos / (dist_km + 0.50)
    df["signed_log_closing"] = slog1p(closing)
    df["signed_log_along"] = slog1p(along)
    df["signed_log_radial"] = slog1p(radial)
    df["signed_log_cross"] = slog1p(cross)
    df["signed_log_dist_change"] = slog1p(dist_change_close)
    df["signed_log_accel"] = slog1p(accel_close)
    df["signed_log_centroid_speed"] = slog1p(cent_speed)
    df["log_growth_pressure"] = np.log1p(np.maximum(df["growth_pressure"], 0.0))
    df["log_threat_gravity"] = np.log1p(np.maximum(df["threat_gravity"], 0.0))
    df["log_threat_momentum"] = np.log1p(np.maximum(df["threat_momentum"], 0.0))
    df["dist_times_alignment"] = dist_km * align_abs
    df["area_times_alignment"] = log_area0 * align_abs
    df["rate_times_alignment"] = np.log1p(np.maximum(grow_rate, 0.0)) * align_abs
    df["approach_minus_cross"] = (close_pos + along_pos) - np.abs(cross)
    df["wavefront_minus_dist"] = wavefront_speed - dist_km * 100.0
    df["dist_over_quality_close"] = d / (1.0 + slope_pos * (0.5 + 0.5 * r2))
    df["potential_12h_gap_km"] = (d - 12.0 * wavefront_speed) / 1000.0
    df["potential_24h_gap_km"] = (d - 24.0 * wavefront_speed) / 1000.0
    df["potential_48h_gap_km"] = (d - 48.0 * wavefront_speed) / 1000.0
    df["potential_72h_gap_km"] = (d - 72.0 * wavefront_speed) / 1000.0
    df["close_accel_interaction"] = close_pos * accel_pos / (dist_km + 1.0)
    df["r2_close_interaction"] = r2 * close_pos
    df["r2_wave_interaction"] = r2 * wavefront_speed
    df["r2_growth_interaction"] = r2 * np.log1p(np.maximum(grow_rate, 0.0))
    df["growth_per_dist"] = np.log1p(np.maximum(grow_abs, 0.0)) / (dist_km + 0.5)
    df["area_per_dist"] = log_area0 / (dist_km + 0.5)
    df["coverage_x_quality"] = df["coverage_density"] * (0.5 + 0.5 * r2)
    df["spread_vector_strength"] = np.sqrt(close_pos ** 2 + along_pos ** 2 + radial_pos ** 2)
    df["spread_vector_per_dist"] = df["spread_vector_strength"] / (dist_km + 0.5)
    df["approach_signal"] = 0.5 * close_pos + 0.3 * along_pos + 0.2 * slope_pos
    df["approach_signal_per_dist"] = df["approach_signal"] / (dist_km + 0.5)
    df["eta_rank_proxy"] = 1.0 / (1.0 + df["eta_hmean_h"])

    for km in [3, 5, 10, 20]:
        df[f"is_within_{km}km"] = (dist_km <= km).astype(float)
    for h in HORIZONS:
        df[f"eta_min_le_{h}h"] = (df["eta_min_h"] <= h).astype(float)
        df[f"eta_wave_le_{h}h"] = (df["eta_wave_h"] <= h).astype(float)

    num_cols = [c for c in df.columns if c not in ["event_id"]]
    for c in num_cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


class IdentityCalibrator:
    def fit(self, x, y):
        return self
    def predict(self, x):
        x = np.asarray(x, dtype=float)
        return np.clip(x, 0.0, 1.0)


class MeanCalibrator:
    def fit(self, x, y):
        y = np.asarray(y, dtype=float)
        self.p = float(np.mean(y)) if len(y) else 0.5
        return self
    def predict(self, x):
        x = np.asarray(x, dtype=float)
        return np.full_like(x, self.p, dtype=float)


class OneCalibrator:
    def fit(self, x, y):
        return self
    def predict(self, x):
        x = np.asarray(x, dtype=float)
        return np.ones_like(x, dtype=float)


class PlattCalibrator:
    def __init__(self, C=1.0):
        self.C = C
        self.model = LogisticRegression(C=C, solver="lbfgs", max_iter=1000)
    def fit(self, x, y):
        xx = np.clip(np.asarray(x, dtype=float), 1e-6, 1 - 1e-6)
        yy = np.asarray(y, dtype=int)
        self.model.fit(logit(xx).reshape(-1, 1), yy)
        return self
    def predict(self, x):
        xx = np.clip(np.asarray(x, dtype=float), 1e-6, 1 - 1e-6)
        return self.model.predict_proba(logit(xx).reshape(-1, 1))[:, 1]


class BetaCalibrator:
    def __init__(self, C=1.0):
        self.C = C
        self.model = LogisticRegression(C=C, solver="lbfgs", max_iter=1000)
    def fit(self, x, y):
        xx = np.clip(np.asarray(x, dtype=float), 1e-6, 1 - 1e-6)
        feats = np.column_stack([np.log(xx), np.log1p(-xx)])
        self.model.fit(feats, np.asarray(y, dtype=int))
        return self
    def predict(self, x):
        xx = np.clip(np.asarray(x, dtype=float), 1e-6, 1 - 1e-6)
        feats = np.column_stack([np.log(xx), np.log1p(-xx)])
        return self.model.predict_proba(feats)[:, 1]


class IsotonicCalibrator:
    def __init__(self):
        self.model = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    def fit(self, x, y):
        self.model.fit(np.asarray(x, dtype=float), np.asarray(y, dtype=float))
        return self
    def predict(self, x):
        return np.clip(self.model.predict(np.asarray(x, dtype=float)), 0.0, 1.0)


def choose_calibrator(oof_col, cache_obj, seed=42):
    mask = cache_obj["mask"]
    y = cache_obj["y"]
    x = np.asarray(oof_col, dtype=float)[mask]
    y = np.asarray(y, dtype=int)[mask]

    if len(x) < 12 or len(np.unique(y)) < 2:
        return IdentityCalibrator(), {"name": "identity", "cv_brier": np.nan}

    pos = int(y.sum())
    neg = int(len(y) - pos)
    n_splits = min(5, pos, neg)
    if n_splits < 2:
        return IdentityCalibrator(), {"name": "identity", "cv_brier": np.nan}

    candidates = {
        "identity": IdentityCalibrator,
        "mean": MeanCalibrator,
        "one": OneCalibrator,
        "platt": PlattCalibrator,
        "beta": BetaCalibrator,
        "isotonic": IsotonicCalibrator,
    }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    best_name = None
    best_score = np.inf

    for name, cls in candidates.items():
        scores = []
        ok = True
        for tr, va in skf.split(np.zeros(len(x)), y):
            try:
                model = cls()
                model.fit(x[tr], y[tr])
                p = model.predict(x[va])
                scores.append(np.mean((p - y[va]) ** 2))
            except Exception:
                ok = False
                break
        if ok and scores:
            score = float(np.mean(scores))
            if score < best_score:
                best_score = score
                best_name = name

    if best_name is None:
        best_name = "identity"

    model = candidates[best_name]()
    model.fit(x, y)
    return model, {"name": best_name, "cv_brier": float(best_score) if np.isfinite(best_score) else np.nan}


def map_rank_to_distribution(base_col, rank_signal):
    base_col = np.asarray(base_col, dtype=float)
    rank_signal = np.asarray(rank_signal, dtype=float)
    order = np.argsort(rank_signal, kind="mergesort")
    sorted_base = np.sort(base_col)
    out = np.empty_like(sorted_base)
    out[order] = sorted_base
    return out


def pct_rank(x):
    return pd.Series(np.asarray(x, dtype=float)).rank(method="average", pct=True).to_numpy(dtype=float)


def weighted_quantile(values, weights, q):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    if len(values) == 0:
        return np.nan
    sorter = np.argsort(values)
    values = values[sorter]
    weights = np.clip(weights[sorter], 0.0, None)
    if weights.sum() <= 0:
        weights = np.ones_like(weights, dtype=float)
    cum = np.cumsum(weights) / np.sum(weights)
    return float(np.interp(q, cum, values))


def hash_frame(df):
    vals = np.round(df[REQUIRED_COLS[1:]].to_numpy(dtype=float), 10)
    return hashlib.md5(vals.tobytes()).hexdigest()


def extract_submission_index(path):
    m = re.search(r"sub(?:mission)?[_-]?(\d+)", os.path.basename(path).lower())
    return int(m.group(1)) if m else None


def extract_score_from_name(path, lookup=None):
    fn = os.path.basename(path)
    m = re.search(r"([0-9]\.[0-9]{4,5})", fn)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            pass
    idx = extract_submission_index(fn)
    if lookup and idx in lookup:
        return float(lookup[idx])
    return None


def safe_read_submission_csv(path, sample_ids):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if "event_id" not in df.columns:
        return None
    for c in REQUIRED_COLS[1:]:
        if c not in df.columns:
            return None
    df = df[REQUIRED_COLS].copy()
    if df["event_id"].duplicated().any():
        return None
    try:
        df = df.set_index("event_id").reindex(sample_ids).reset_index()
    except Exception:
        return None
    if df["event_id"].isna().any():
        return None
    vals = df[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if not np.isfinite(vals).all():
        return None
    vals = enforce_monotonicity(np.clip(vals, 0.0, 1.0))
    df.loc[:, REQUIRED_COLS[1:]] = vals
    return df


def weighted_na_probs(times, events, weights, horizons=HORIZONS):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    weights = np.clip(np.asarray(weights, dtype=float), 0.0, None)
    if len(times) == 0 or weights.sum() <= 0:
        return np.full(len(horizons), 0.5, dtype=float)

    event_times = np.sort(np.unique(times[events == 1]))
    if len(event_times) == 0:
        return np.zeros(len(horizons), dtype=float)

    increments = []
    for t in event_times:
        d = weights[(times == t) & (events == 1)].sum()
        r = weights[times >= t].sum()
        increments.append(0.0 if r <= 0 else d / r)
    increments = np.asarray(increments, dtype=float)
    cum = np.cumsum(increments)

    probs = []
    for h in horizons:
        idx = np.searchsorted(event_times, h, side="right") - 1
        ch = float(cum[idx]) if idx >= 0 else 0.0
        probs.append(1.0 - math.exp(-ch))
    return np.asarray(probs, dtype=float)


def make_discrete_rows(X_df, time_vals, event_vals):
    rows = []
    ys = []
    ws = []
    interval_ids = []
    interval_weights = [1.00, 1.15, 1.05, 0.35]

    for i in range(len(X_df)):
        t = float(time_vals[i])
        e = int(event_vals[i])
        for j, (start, end) in enumerate(INTERVALS):
            if t <= start:
                break
            if e == 1 and t <= end:
                rows.append(i)
                ys.append(1)
                ws.append(interval_weights[j])
                interval_ids.append(j)
                break
            if e == 0 and t < end:
                break
            rows.append(i)
            ys.append(0)
            ws.append(interval_weights[j])
            interval_ids.append(j)

    X_long = X_df.iloc[rows].copy().reset_index(drop=True)
    X_long["interval_id"] = np.asarray(interval_ids, dtype=int)
    X_long["interval_start"] = [INTERVALS[j][0] for j in interval_ids]
    X_long["interval_end"] = [INTERVALS[j][1] for j in interval_ids]
    X_long["interval_len"] = [INTERVALS[j][1] - INTERVALS[j][0] for j in interval_ids]
    for j in range(len(INTERVALS)):
        X_long[f"interval_{j}"] = (X_long["interval_id"] == j).astype(int)
    y_long = np.asarray(ys, dtype=int)
    w_long = np.asarray(ws, dtype=float) * class_balance_weights(y_long, power=0.60, max_pos_weight=5.0)
    return X_long, y_long, w_long


def add_interval_features(X_df, interval_id):
    X = X_df.copy()
    X["interval_id"] = int(interval_id)
    X["interval_start"] = INTERVALS[interval_id][0]
    X["interval_end"] = INTERVALS[interval_id][1]
    X["interval_len"] = INTERVALS[interval_id][1] - INTERVALS[interval_id][0]
    for j in range(len(INTERVALS)):
        X[f"interval_{j}"] = 1 if j == interval_id else 0
    return X


def predict_discrete_hazards(model, X_df):
    hazards = []
    for j in range(len(INTERVALS)):
        Xj = add_interval_features(X_df, j)
        p = model.predict_proba(Xj)[:, 1]
        hazards.append(np.clip(p, 1e-6, 1 - 1e-6))
    hazards = np.column_stack(hazards)
    surv = np.cumprod(1.0 - hazards, axis=1)
    probs = 1.0 - surv
    return enforce_monotonicity(probs)


def fit_predict_xgb_aft(X_tr, t_tr, e_tr, X_va, X_te, cfg):
    if np.asarray(e_tr, dtype=int).sum() == 0:
        base = weighted_na_probs(t_tr, e_tr, np.ones(len(t_tr), dtype=float), horizons=HORIZONS)
        return np.tile(base, (len(X_va), 1)), np.tile(base, (len(X_te), 1))
    dtr = xgb.DMatrix(X_tr)
    dva = xgb.DMatrix(X_va)
    dte = xgb.DMatrix(X_te)

    lb = np.asarray(t_tr, dtype=float).copy()
    ub = np.asarray(t_tr, dtype=float).copy()
    ub[np.asarray(e_tr, dtype=int) == 0] = np.inf

    dtr.set_float_info("label_lower_bound", lb)
    dtr.set_float_info("label_upper_bound", ub)

    params = {
        "objective": "survival:aft",
        "eval_metric": "aft-nloglik",
        "aft_loss_distribution": cfg["dist"],
        "aft_loss_distribution_scale": cfg["scale"],
        "tree_method": "hist",
        "learning_rate": cfg["learning_rate"],
        "max_depth": cfg["max_depth"],
        "min_child_weight": cfg["min_child_weight"],
        "subsample": cfg["subsample"],
        "colsample_bytree": cfg["colsample_bytree"],
        "reg_alpha": cfg["reg_alpha"],
        "reg_lambda": cfg["reg_lambda"],
        "seed": cfg["seed"],
        "verbosity": 0,
    }
    evals = []
    if len(X_va) > 0:
        lb_va = np.full(len(X_va), np.nan)
        ub_va = np.full(len(X_va), np.nan)
        # valid labels are not used except for early stopping if provided;
        # we skip eval_set to avoid passing labels inconsistently.
    bst = xgb.train(
        params,
        dtr,
        num_boost_round=cfg["num_boost_round"],
        verbose_eval=False,
    )
    mu_va = bst.predict(dva)
    mu_te = bst.predict(dte)
    p_va = []
    p_te = []
    for h in HORIZONS:
        zv = (np.log(h) - mu_va) / cfg["scale"]
        zt = (np.log(h) - mu_te) / cfg["scale"]
        p_va.append(aft_cdf(zv, cfg["dist"]))
        p_te.append(aft_cdf(zt, cfg["dist"]))
    return enforce_monotonicity(np.column_stack(p_va)), enforce_monotonicity(np.column_stack(p_te))


def fit_predict_xgb_cox(X_tr, t_tr, e_tr, X_va, X_te, cfg):
    if np.asarray(e_tr, dtype=int).sum() == 0:
        base = weighted_na_probs(t_tr, e_tr, np.ones(len(t_tr), dtype=float), horizons=HORIZONS)
        return np.tile(base, (len(X_va), 1)), np.tile(base, (len(X_te), 1))
    y_tr = np.where(np.asarray(e_tr, dtype=int) == 1, np.asarray(t_tr, dtype=float), -np.asarray(t_tr, dtype=float))
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dva = xgb.DMatrix(X_va)
    dte = xgb.DMatrix(X_te)

    params = {
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "tree_method": "hist",
        "learning_rate": cfg["learning_rate"],
        "max_depth": cfg["max_depth"],
        "min_child_weight": cfg["min_child_weight"],
        "subsample": cfg["subsample"],
        "colsample_bytree": cfg["colsample_bytree"],
        "reg_alpha": cfg["reg_alpha"],
        "reg_lambda": cfg["reg_lambda"],
        "seed": cfg["seed"],
        "verbosity": 0,
    }
    bst = xgb.train(
        params,
        dtr,
        num_boost_round=cfg["num_boost_round"],
        verbose_eval=False,
    )

    hr_tr = np.clip(bst.predict(dtr), 1e-12, None)
    hr_va = np.clip(bst.predict(dva), 1e-12, None)
    hr_te = np.clip(bst.predict(dte), 1e-12, None)

    base_haz = breslow_cumhaz(t_tr, e_tr, hr_tr, horizons=HORIZONS)
    p_va = []
    p_te = []
    for h in HORIZONS:
        H0 = base_haz[h]
        p_va.append(1.0 - np.exp(-H0 * hr_va))
        p_te.append(1.0 - np.exp(-H0 * hr_te))
    return enforce_monotonicity(np.column_stack(p_va)), enforce_monotonicity(np.column_stack(p_te))


def fit_predict_cat_horizons(X_tr_df, t_tr, e_tr, X_va_df, X_te_df, cfg):
    valid = []
    test = []
    for h in HORIZONS:
        mask_tr = horizon_train_mask(t_tr, e_tr, h)
        y_tr = horizon_target(t_tr, e_tr, h)[mask_tr]
        Xh_tr = X_tr_df.loc[mask_tr].copy()
        if len(np.unique(y_tr)) < 2:
            const_p = float(np.mean(y_tr)) if len(y_tr) else 0.5
            valid.append(np.full(len(X_va_df), const_p, dtype=float))
            test.append(np.full(len(X_te_df), const_p, dtype=float))
            continue
        w_tr = class_balance_weights(y_tr, power=0.70, max_pos_weight=6.0)

        model = CatBoostClassifier(
            iterations=cfg["iterations"],
            learning_rate=cfg["learning_rate"],
            depth=cfg["depth"],
            l2_leaf_reg=cfg["l2_leaf_reg"],
            random_strength=cfg["random_strength"],
            bagging_temperature=cfg["bagging_temperature"],
            loss_function="Logloss",
            eval_metric="Logloss",
            verbose=False,
            random_seed=cfg["seed"] + int(h),
            allow_writing_files=False,
            thread_count=MAX_THREADS,
        )
        model.fit(Xh_tr, y_tr, sample_weight=w_tr)
        valid.append(model.predict_proba(X_va_df)[:, 1])
        test.append(model.predict_proba(X_te_df)[:, 1])
    return enforce_monotonicity(np.column_stack(valid)), enforce_monotonicity(np.column_stack(test))


def fit_predict_lgb_horizons(X_tr_df, t_tr, e_tr, X_va_df, X_te_df, cfg):
    valid = []
    test = []
    for h in HORIZONS:
        mask_tr = horizon_train_mask(t_tr, e_tr, h)
        y_tr = horizon_target(t_tr, e_tr, h)[mask_tr]
        Xh_tr = X_tr_df.loc[mask_tr].copy()
        if len(np.unique(y_tr)) < 2:
            const_p = float(np.mean(y_tr)) if len(y_tr) else 0.5
            valid.append(np.full(len(X_va_df), const_p, dtype=float))
            test.append(np.full(len(X_te_df), const_p, dtype=float))
            continue
        w_tr = class_balance_weights(y_tr, power=0.70, max_pos_weight=6.0)

        model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            num_leaves=cfg["num_leaves"],
            max_depth=cfg["max_depth"],
            min_child_samples=cfg["min_child_samples"],
            subsample=cfg["subsample"],
            colsample_bytree=cfg["colsample_bytree"],
            reg_alpha=cfg["reg_alpha"],
            reg_lambda=cfg["reg_lambda"],
            random_state=cfg["seed"] + int(h),
            n_jobs=MAX_THREADS,
            verbosity=-1,
        )
        model.fit(Xh_tr, y_tr, sample_weight=w_tr)
        valid.append(model.predict_proba(X_va_df)[:, 1])
        test.append(model.predict_proba(X_te_df)[:, 1])
    return enforce_monotonicity(np.column_stack(valid)), enforce_monotonicity(np.column_stack(test))


def fit_predict_et_horizons(X_tr_df, t_tr, e_tr, X_va_df, X_te_df, cfg):
    valid = []
    test = []
    for h in HORIZONS:
        mask_tr = horizon_train_mask(t_tr, e_tr, h)
        y_tr = horizon_target(t_tr, e_tr, h)[mask_tr]
        Xh_tr = X_tr_df.loc[mask_tr].copy()
        if len(np.unique(y_tr)) < 2:
            const_p = float(np.mean(y_tr)) if len(y_tr) else 0.5
            valid.append(np.full(len(X_va_df), const_p, dtype=float))
            test.append(np.full(len(X_te_df), const_p, dtype=float))
            continue
        w_tr = class_balance_weights(y_tr, power=0.55, max_pos_weight=5.0)

        model = ExtraTreesClassifier(
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            min_samples_leaf=cfg["min_samples_leaf"],
            max_features=cfg["max_features"],
            class_weight="balanced_subsample",
            random_state=cfg["seed"] + int(h),
            n_jobs=MAX_THREADS,
        )
        model.fit(Xh_tr, y_tr, sample_weight=w_tr)
        valid.append(model.predict_proba(X_va_df)[:, 1])
        test.append(model.predict_proba(X_te_df)[:, 1])
    return enforce_monotonicity(np.column_stack(valid)), enforce_monotonicity(np.column_stack(test))


def fit_predict_cat_discrete(X_tr_df, t_tr, e_tr, X_va_df, X_te_df, cfg):
    X_long, y_long, w_long = make_discrete_rows(X_tr_df, t_tr, e_tr)
    if len(np.unique(y_long)) < 2:
        const_h = float(np.mean(y_long)) if len(y_long) else 0.5
        const_prob = np.array([1-(1-const_h)**(i+1) for i in range(len(HORIZONS))], dtype=float)
        return np.tile(const_prob, (len(X_va_df), 1)), np.tile(const_prob, (len(X_te_df), 1))
    model = CatBoostClassifier(
        iterations=cfg["iterations"],
        learning_rate=cfg["learning_rate"],
        depth=cfg["depth"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        random_strength=cfg["random_strength"],
        bagging_temperature=cfg["bagging_temperature"],
        loss_function="Logloss",
        eval_metric="Logloss",
        verbose=False,
        random_seed=cfg["seed"],
        allow_writing_files=False,
        thread_count=MAX_THREADS,
    )
    model.fit(X_long, y_long, sample_weight=w_long)
    return predict_discrete_hazards(model, X_va_df), predict_discrete_hazards(model, X_te_df)


def fit_predict_phys_logit(X_tr_df, t_tr, e_tr, X_va_df, X_te_df, feat_cols, cfg):
    valid = []
    test = []
    for h in HORIZONS:
        mask_tr = horizon_train_mask(t_tr, e_tr, h)
        y_tr = horizon_target(t_tr, e_tr, h)[mask_tr]
        Xh_tr = X_tr_df.loc[mask_tr, feat_cols].copy()

        if len(np.unique(y_tr)) < 2:
            const_p = float(np.mean(y_tr)) if len(y_tr) else 0.5
            valid.append(np.full(len(X_va_df), const_p, dtype=float))
            test.append(np.full(len(X_te_df), const_p, dtype=float))
            continue

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            ("poly", PolynomialFeatures(degree=cfg["degree"], include_bias=False)),
            ("model", LogisticRegression(
                C=cfg["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=4000,
                random_state=cfg["seed"] + int(h),
            ))
        ])
        w_tr = class_balance_weights(y_tr, power=0.60, max_pos_weight=6.0)
        pipe.fit(Xh_tr, y_tr, model__sample_weight=w_tr)
        valid.append(pipe.predict_proba(X_va_df[feat_cols])[:, 1])
        test.append(pipe.predict_proba(X_te_df[feat_cols])[:, 1])
    return enforce_monotonicity(np.column_stack(valid)), enforce_monotonicity(np.column_stack(test))


def fit_predict_knn_survival(X_tr_df, t_tr, e_tr, X_va_df, X_te_df, feat_cols, cfg):
    imp = SimpleImputer(strategy="median")
    scl = RobustScaler()
    Xtr = scl.fit_transform(imp.fit_transform(X_tr_df[feat_cols]))
    Xva = scl.transform(imp.transform(X_va_df[feat_cols]))
    Xte = scl.transform(imp.transform(X_te_df[feat_cols]))

    global_p = weighted_na_probs(t_tr, e_tr, np.ones(len(t_tr), dtype=float), horizons=HORIZONS)

    def _predict(Xpred):
        out = np.zeros((len(Xpred), len(HORIZONS)), dtype=float)
        for i, x in enumerate(Xpred):
            d = np.sqrt(np.sum((Xtr - x) ** 2, axis=1))
            order = np.argsort(d)
            k = min(cfg["k"], len(order))
            idx = order[:k]
            dd = d[idx]
            bw = max(np.percentile(dd, 80), 1e-6)
            w = np.exp(-cfg["beta"] * (dd / bw) ** 2)
            p_local = weighted_na_probs(np.asarray(t_tr)[idx], np.asarray(e_tr)[idx], w, horizons=HORIZONS)
            eff_n = (w.sum() ** 2) / max((w ** 2).sum(), 1e-12)
            lam = float(np.clip((eff_n - 3.0) / max(k - 3.0, 1.0), 0.30, 0.95))
            out[i] = lam * p_local + (1.0 - lam) * global_p
        return enforce_monotonicity(out)

    return _predict(Xva), _predict(Xte)


def optimize_greedy_blend(preds_dict, candidate_names, metric_func, max_add=8, seed=42):
    candidate_names = list(candidate_names)
    if len(candidate_names) == 0:
        raise ValueError("No candidates to blend")
    base_scores = {n: metric_func(preds_dict[n]) for n in candidate_names}
    best_name = max(candidate_names, key=lambda n: base_scores[n])
    current = preds_dict[best_name].copy()
    weights = {best_name: 1.0}
    best_score = base_scores[best_name]
    remaining = [n for n in candidate_names if n != best_name]
    alpha_grid = [0.03, 0.05, 0.07, 0.10, 0.13, 0.17, 0.22, 0.28, 0.35, 0.45]
    history = []

    for _ in range(max_add):
        step_best = None
        for name in remaining:
            cand = preds_dict[name]
            for a in alpha_grid:
                pred = enforce_monotonicity((1.0 - a) * current + a * cand)
                s = metric_func(pred)
                if step_best is None or s > step_best[0] + 1e-12:
                    step_best = (s, name, a, pred)
        if step_best is None or step_best[0] <= best_score + 1e-6:
            break
        s, name, a, pred = step_best
        current = pred
        best_score = s
        for k in list(weights.keys()):
            weights[k] *= (1.0 - a)
        weights[name] = weights.get(name, 0.0) + a
        remaining.remove(name)
        history.append({"name": name, "alpha": float(a), "score": float(s)})

    sel_names = list(weights.keys())
    arr = np.stack([preds_dict[n] for n in sel_names], axis=0)
    w = np.asarray([weights[n] for n in sel_names], dtype=float)
    w /= w.sum()

    rng = np.random.default_rng(seed)
    best_w = w.copy()
    best_pred = enforce_monotonicity(np.tensordot(best_w, arr, axes=(0, 0)))
    best_score = metric_func(best_pred)

    for _ in range(3000):
        noise = np.exp(rng.normal(0.0, 0.35, size=len(best_w)))
        w_try = best_w * noise
        w_try = np.clip(w_try, 1e-8, None)
        w_try /= w_try.sum()
        pred = enforce_monotonicity(np.tensordot(w_try, arr, axes=(0, 0)))
        s = metric_func(pred)
        if s > best_score + 1e-8:
            best_score = s
            best_w = w_try
            best_pred = pred

    final_weights = dict(zip(sel_names, best_w))
    return best_pred, final_weights, best_score, history


def optimize_single_horizon(preds_dict, candidate_names, target_col, score_func, max_add=6):
    candidate_names = list(candidate_names)
    if len(candidate_names) == 0:
        raise ValueError("No candidates for single horizon optimization")
    base_scores = {n: score_func(preds_dict[n][:, target_col]) for n in candidate_names}
    best_name = max(candidate_names, key=lambda n: base_scores[n])
    current = preds_dict[best_name][:, target_col].copy()
    weights = {best_name: 1.0}
    best_score = base_scores[best_name]
    remaining = [n for n in candidate_names if n != best_name]
    alpha_grid = [0.03, 0.05, 0.07, 0.10, 0.13, 0.17, 0.22, 0.28, 0.35, 0.45]

    for _ in range(max_add):
        step_best = None
        for name in remaining:
            cand = preds_dict[name][:, target_col]
            for a in alpha_grid:
                pred = np.clip((1.0 - a) * current + a * cand, 0.0, 1.0)
                s = score_func(pred)
                if step_best is None or s > step_best[0] + 1e-12:
                    step_best = (s, name, a, pred)
        if step_best is None or step_best[0] <= best_score + 1e-6:
            break
        s, name, a, pred = step_best
        current = pred
        best_score = s
        for k in list(weights.keys()):
            weights[k] *= (1.0 - a)
        weights[name] = weights.get(name, 0.0) + a
        remaining.remove(name)

    sel_names = list(weights.keys())
    arr = np.stack([preds_dict[n][:, target_col] for n in sel_names], axis=0)
    w = np.asarray([weights[n] for n in sel_names], dtype=float)
    w /= w.sum()

    rng = np.random.default_rng(SEED + 13 + target_col)
    best_w = w.copy()
    best_pred = np.clip(np.tensordot(best_w, arr, axes=(0, 0)), 0.0, 1.0)
    best_score = score_func(best_pred)
    for _ in range(2000):
        noise = np.exp(rng.normal(0.0, 0.35, size=len(best_w)))
        w_try = np.clip(best_w * noise, 1e-8, None)
        w_try /= w_try.sum()
        pred = np.clip(np.tensordot(w_try, arr, axes=(0, 0)), 0.0, 1.0)
        s = score_func(pred)
        if s > best_score + 1e-8:
            best_score = s
            best_w = w_try
            best_pred = pred
    return best_pred, dict(zip(sel_names, best_w)), best_score


dataset_root = find_core_dataset()
print("Core dataset:", dataset_root)

train = pd.read_csv(os.path.join(dataset_root, "train.csv"))
test = pd.read_csv(os.path.join(dataset_root, "test.csv"))
sample_sub = pd.read_csv(os.path.join(dataset_root, "sample_submission.csv"))

train = engineer_features(train)
test = engineer_features(test)

feature_cols = [c for c in train.columns if c not in ["event_id", "time_to_hit_hours", "event"]]
for c in feature_cols:
    if c not in test.columns:
        test[c] = np.nan
test = test[["event_id"] + feature_cols].copy()

X_train_df = train[feature_cols].replace([np.inf, -np.inf], np.nan)
X_test_df = test[feature_cols].replace([np.inf, -np.inf], np.nan)

median_vals = X_train_df.median(numeric_only=True)
X_train_df = X_train_df.fillna(median_vals)
X_test_df = X_test_df.fillna(median_vals)

time_train = train["time_to_hit_hours"].astype(float).to_numpy()
event_train = train["event"].astype(int).to_numpy()

cv_splits = build_cv_splits(train, n_splits=5, seeds=(42, 202, 2026))
pair_left, pair_right = build_cindex_pairs(time_train, event_train)
brier_cache = build_brier_cache(time_train, event_train, horizons=HORIZONS)

physics_feature_candidates = [
    "dist_min_ci_0_5h", "dist_km", "log_dist_m", "inv_dist_km", "closing_speed_m_per_h",
    "along_track_speed", "radial_growth_rate_m_per_h", "dist_slope_ci_0_5h", "dist_change_ci_0_5h",
    "dist_fit_r2_0_5h", "alignment_abs", "alignment_cos", "projected_advance_m", "advance_ratio",
    "area_first_ha", "log1p_area_first", "area_growth_rate_ha_per_h", "log1p_growth",
    "threat_gravity", "threat_momentum", "motion_pressure", "size_pressure",
    "eta_min_h", "eta_wave_h", "eta_toward_h", "eta_hmean_h",
    "potential_12h_gap_km", "potential_24h_gap_km", "potential_48h_gap_km",
    "low_temporal_resolution_0_5h", "num_perimeters_0_5h", "dt_first_last_0_5h",
    "event_start_hour_sin", "event_start_hour_cos", "event_start_month_sin", "event_start_month_cos",
]
physics_feats = [c for c in physics_feature_candidates if c in X_train_df.columns]

knn_feature_sets = []
set1 = [c for c in [
    "dist_min_ci_0_5h", "closing_speed_m_per_h", "along_track_speed", "radial_growth_rate_m_per_h",
    "alignment_abs", "projected_advance_m", "dist_fit_r2_0_5h", "area_first_ha",
    "area_growth_rate_ha_per_h", "low_temporal_resolution_0_5h",
] if c in X_train_df.columns]
set2 = [c for c in [
    "dist_km", "log_dist_m", "threat_gravity", "threat_momentum", "eta_min_h", "eta_wave_h",
    "advance_ratio", "motion_pressure", "size_pressure", "coverage_density",
    "event_start_hour_sin", "event_start_hour_cos", "event_start_month_sin", "event_start_month_cos"
] if c in X_train_df.columns]
if len(set1) >= 5:
    knn_feature_sets.append(("knnset1", set1))
if len(set2) >= 5:
    knn_feature_sets.append(("knnset2", set2))

AFT_CONFIGS = [
    {"name": "aft__normal_d3_s08", "dist": "normal", "scale": 0.80, "learning_rate": 0.035, "max_depth": 3, "min_child_weight": 3.0, "subsample": 0.88, "colsample_bytree": 0.82, "reg_alpha": 0.05, "reg_lambda": 5.0, "num_boost_round": 420},
    {"name": "aft__normal_d4_s12", "dist": "normal", "scale": 1.20, "learning_rate": 0.030, "max_depth": 4, "min_child_weight": 4.0, "subsample": 0.80, "colsample_bytree": 0.78, "reg_alpha": 0.10, "reg_lambda": 8.0, "num_boost_round": 480},
    {"name": "aft__logistic_d3_s10", "dist": "logistic", "scale": 1.00, "learning_rate": 0.035, "max_depth": 3, "min_child_weight": 3.0, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.05, "reg_lambda": 4.0, "num_boost_round": 420},
    {"name": "aft__logistic_d2_s16", "dist": "logistic", "scale": 1.60, "learning_rate": 0.045, "max_depth": 2, "min_child_weight": 2.0, "subsample": 0.92, "colsample_bytree": 0.72, "reg_alpha": 0.00, "reg_lambda": 3.5, "num_boost_round": 360},
    {"name": "aft__extreme_d3_s10", "dist": "extreme", "scale": 1.00, "learning_rate": 0.035, "max_depth": 3, "min_child_weight": 3.0, "subsample": 0.86, "colsample_bytree": 0.84, "reg_alpha": 0.05, "reg_lambda": 4.5, "num_boost_round": 420},
]

COX_CONFIGS = [
    {"name": "cox__d3_lr035", "learning_rate": 0.035, "max_depth": 3, "min_child_weight": 3.0, "subsample": 0.88, "colsample_bytree": 0.82, "reg_alpha": 0.05, "reg_lambda": 5.0, "num_boost_round": 420},
    {"name": "cox__d4_lr030", "learning_rate": 0.030, "max_depth": 4, "min_child_weight": 4.0, "subsample": 0.82, "colsample_bytree": 0.78, "reg_alpha": 0.10, "reg_lambda": 7.0, "num_boost_round": 500},
    {"name": "cox__d2_lr050", "learning_rate": 0.050, "max_depth": 2, "min_child_weight": 2.0, "subsample": 0.94, "colsample_bytree": 0.70, "reg_alpha": 0.00, "reg_lambda": 3.0, "num_boost_round": 320},
]

CAT_HORIZON_CONFIGS = [
    {"name": "catH__d4_lr03", "iterations": 1600, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 12.0, "random_strength": 1.0, "bagging_temperature": 1.0},
    {"name": "catH__d5_lr025", "iterations": 2200, "learning_rate": 0.025, "depth": 5, "l2_leaf_reg": 18.0, "random_strength": 1.2, "bagging_temperature": 1.4},
]

LGB_HORIZON_CONFIGS = [
    {"name": "lgbH__nl15_lr03", "n_estimators": 1200, "learning_rate": 0.030, "num_leaves": 15, "max_depth": -1, "min_child_samples": 10, "subsample": 0.85, "colsample_bytree": 0.82, "reg_alpha": 0.05, "reg_lambda": 4.0},
    {"name": "lgbH__nl23_lr025", "n_estimators": 1600, "learning_rate": 0.025, "num_leaves": 23, "max_depth": -1, "min_child_samples": 14, "subsample": 0.78, "colsample_bytree": 0.76, "reg_alpha": 0.10, "reg_lambda": 6.0},
]

ET_HORIZON_CONFIGS = [
    {"name": "etH__800_md10", "n_estimators": 800, "max_depth": 10, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"name": "etH__1200_mdNone", "n_estimators": 1200, "max_depth": None, "min_samples_leaf": 2, "max_features": 0.65},
]

CAT_DISCRETE_CONFIGS = [
    {"name": "catD__d4_lr03", "iterations": 1800, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 12.0, "random_strength": 1.0, "bagging_temperature": 1.0},
    {"name": "catD__d5_lr025", "iterations": 2400, "learning_rate": 0.025, "depth": 5, "l2_leaf_reg": 18.0, "random_strength": 1.1, "bagging_temperature": 1.3},
]

PHYS_LOGIT_CONFIGS = [
    {"name": "phys__deg1_c08", "degree": 1, "C": 0.80},
    {"name": "phys__deg2_c02", "degree": 2, "C": 0.20},
]

KNN_CONFIGS = []
for base_name, _cols in knn_feature_sets:
    KNN_CONFIGS.extend([
        {"name": f"knn__{base_name}_k14_b12", "k": 14, "beta": 1.2, "feat_name": base_name},
        {"name": f"knn__{base_name}_k24_b16", "k": 24, "beta": 1.6, "feat_name": base_name},
    ])

models_to_run = []

if HAS_XGB:
    for cfg in AFT_CONFIGS:
        models_to_run.append(("aft", cfg["name"], cfg))
    for cfg in COX_CONFIGS:
        models_to_run.append(("cox", cfg["name"], cfg))

if HAS_CAT:
    for cfg in CAT_HORIZON_CONFIGS:
        models_to_run.append(("cat_h", cfg["name"], cfg))
    for cfg in CAT_DISCRETE_CONFIGS:
        models_to_run.append(("cat_d", cfg["name"], cfg))

if HAS_LGB:
    for cfg in LGB_HORIZON_CONFIGS:
        models_to_run.append(("lgb_h", cfg["name"], cfg))

for cfg in ET_HORIZON_CONFIGS:
    models_to_run.append(("et_h", cfg["name"], cfg))
for cfg in PHYS_LOGIT_CONFIGS:
    models_to_run.append(("phys", cfg["name"], cfg))
for cfg in KNN_CONFIGS:
    models_to_run.append(("knn", cfg["name"], cfg))

print("Base models scheduled:", len(models_to_run))

model_oof = {}
model_test = {}
model_meta = []

for family, model_name, cfg in models_to_run:
    print(f"\n===== {model_name} ({family}) =====")
    seed_everything(SEED)
    oof_sum = np.zeros((len(train), len(HORIZONS)), dtype=float)
    oof_cnt = np.zeros(len(train), dtype=float)
    test_sum = np.zeros((len(test), len(HORIZONS)), dtype=float)

    for split_id, split in enumerate(cv_splits):
        tr_idx = split["train_idx"]
        va_idx = split["valid_idx"]

        Xtr_df = X_train_df.iloc[tr_idx].copy()
        Xva_df = X_train_df.iloc[va_idx].copy()
        Xte_df = X_test_df.copy()

        ttr = time_train[tr_idx]
        etr = event_train[tr_idx]

        cfg_local = dict(cfg)
        cfg_local["seed"] = split["seed"] * 100 + split["fold"] + SEED

        try:
            if family == "aft":
                pred_va, pred_te = fit_predict_xgb_aft(Xtr_df.values, ttr, etr, Xva_df.values, Xte_df.values, cfg_local)
            elif family == "cox":
                pred_va, pred_te = fit_predict_xgb_cox(Xtr_df.values, ttr, etr, Xva_df.values, Xte_df.values, cfg_local)
            elif family == "cat_h":
                pred_va, pred_te = fit_predict_cat_horizons(Xtr_df, ttr, etr, Xva_df, Xte_df, cfg_local)
            elif family == "lgb_h":
                pred_va, pred_te = fit_predict_lgb_horizons(Xtr_df, ttr, etr, Xva_df, Xte_df, cfg_local)
            elif family == "et_h":
                pred_va, pred_te = fit_predict_et_horizons(Xtr_df, ttr, etr, Xva_df, Xte_df, cfg_local)
            elif family == "cat_d":
                pred_va, pred_te = fit_predict_cat_discrete(Xtr_df, ttr, etr, Xva_df, Xte_df, cfg_local)
            elif family == "phys":
                pred_va, pred_te = fit_predict_phys_logit(Xtr_df, ttr, etr, Xva_df, Xte_df, physics_feats, cfg_local)
            elif family == "knn":
                feat_cols = None
                for nm, cols in knn_feature_sets:
                    if nm == cfg_local["feat_name"]:
                        feat_cols = cols
                        break
                if feat_cols is None:
                    raise ValueError("Missing KNN feature set")
                pred_va, pred_te = fit_predict_knn_survival(Xtr_df, ttr, etr, Xva_df, Xte_df, feat_cols, cfg_local)
            else:
                raise ValueError(f"Unknown family: {family}")
        except Exception as ex:
            print("Skipped split due to error:", ex)
            pred_va = np.tile(np.mean(event_train), (len(va_idx), len(HORIZONS)))
            pred_te = np.tile(np.mean(event_train), (len(test), len(HORIZONS)))

        pred_va = enforce_monotonicity(np.clip(pred_va, 0.0, 1.0))
        pred_te = enforce_monotonicity(np.clip(pred_te, 0.0, 1.0))

        oof_sum[va_idx] += pred_va
        oof_cnt[va_idx] += 1.0
        test_sum += pred_te

        if (split_id + 1) % 5 == 0:
            gc.collect()

    oof_pred = oof_sum / np.maximum(oof_cnt[:, None], 1.0)
    test_pred = test_sum / max(len(cv_splits), 1)

    oof_pred = enforce_monotonicity(np.clip(oof_pred, 0.0, 1.0))
    test_pred = enforce_monotonicity(np.clip(test_pred, 0.0, 1.0))

    model_oof[model_name] = oof_pred
    model_test[model_name] = test_pred

    score = hybrid_score(oof_pred, pair_left, pair_right, brier_cache)
    b12 = brier_from_cache(oof_pred[:, 0], brier_cache[12])
    b24 = brier_from_cache(oof_pred[:, 1], brier_cache[24])
    b48 = brier_from_cache(oof_pred[:, 2], brier_cache[48])
    b72 = brier_from_cache(oof_pred[:, 3], brier_cache[72])

    print(
        f"Hybrid={score:.6f} | "
        f"B12={b12:.6f} B24={b24:.6f} B48={b48:.6f} B72={b72:.6f}"
    )

    model_meta.append({
        "name": model_name,
        "family": family,
        "hybrid": float(score),
        "brier12": float(b12),
        "brier24": float(b24),
        "brier48": float(b48),
        "brier72": float(b72),
    })

meta_df = pd.DataFrame(model_meta).sort_values(["hybrid", "brier24", "brier48"], ascending=[False, True, True]).reset_index(drop=True)
meta_df.to_csv("/kaggle/working/model_scores.csv", index=False)
print("\nTop base models:")
print(meta_df.head(20).to_string(index=False))

best_single = float(meta_df["hybrid"].max())
candidate_names = []
family_counts = defaultdict(int)

for _, row in meta_df.iterrows():
    name = row["name"]
    fam = row["family"]
    if row["hybrid"] < best_single - 0.040 and len(candidate_names) >= 12:
        continue
    if family_counts[fam] >= 3:
        continue
    too_close = False
    for kept in candidate_names:
        mad = float(np.mean(np.abs(model_oof[name][:, :3] - model_oof[kept][:, :3])))
        if mad < 0.0032:
            too_close = True
            break
    if too_close:
        continue
    candidate_names.append(name)
    family_counts[fam] += 1
    if len(candidate_names) >= 18:
        break

if len(candidate_names) < 5:
    candidate_names = meta_df["name"].head(min(10, len(meta_df))).tolist()

print("\nCandidates for main blend:", candidate_names)

metric_func = lambda pred: hybrid_score(pred, pair_left, pair_right, brier_cache)
core_oof, core_weights, core_score, core_history = optimize_greedy_blend(model_oof, candidate_names, metric_func, max_add=8, seed=SEED)
core_test = enforce_monotonicity(np.tensordot(np.asarray([core_weights[n] for n in core_weights]), np.stack([model_test[n] for n in core_weights], axis=0), axes=(0, 0)))

print("\nCore blend score:", round(core_score, 6))
for k, v in sorted(core_weights.items(), key=lambda z: -z[1]):
    print(f"{k:30s} weight={v:.5f}")

# Rank-enhancement layer
sel_names = list(core_weights.keys())
sel_w = np.asarray([core_weights[n] for n in sel_names], dtype=float)
sel_w /= sel_w.sum()

rank_signal_oof = np.zeros_like(core_oof)
rank_signal_test = np.zeros_like(core_test)

for j in range(len(HORIZONS)):
    rank_stack_oof = np.stack([pct_rank(model_oof[n][:, j]) for n in sel_names], axis=0)
    rank_stack_test = np.stack([pct_rank(model_test[n][:, j]) for n in sel_names], axis=0)
    rank_signal_oof[:, j] = np.tensordot(sel_w, rank_stack_oof, axes=(0, 0))
    rank_signal_test[:, j] = np.tensordot(sel_w, rank_stack_test, axes=(0, 0))

rank_map_oof = np.zeros_like(core_oof)
rank_map_test = np.zeros_like(core_test)
for j in range(len(HORIZONS)):
    rank_map_oof[:, j] = map_rank_to_distribution(core_oof[:, j], rank_signal_oof[:, j])
    rank_map_test[:, j] = map_rank_to_distribution(core_test[:, j], rank_signal_test[:, j])

best_rank_alpha = 0.0
best_rank_score = metric_func(core_oof)
for alpha in np.linspace(0.0, 0.30, 13):
    pred_try = enforce_monotonicity((1.0 - alpha) * core_oof + alpha * rank_map_oof)
    s = metric_func(pred_try)
    if s > best_rank_score + 1e-8:
        best_rank_score = s
        best_rank_alpha = float(alpha)

core_oof = enforce_monotonicity((1.0 - best_rank_alpha) * core_oof + best_rank_alpha * rank_map_oof)
core_test = enforce_monotonicity((1.0 - best_rank_alpha) * core_test + best_rank_alpha * rank_map_test)
print("Rank alpha:", best_rank_alpha, "| score:", round(best_rank_score, 6))

# Specialized 12h refinement
score12_func = lambda col: -brier_from_cache(np.asarray(col, dtype=float), brier_cache[12])
top12_names = meta_df.sort_values(["brier12", "hybrid"], ascending=[True, False])["name"].head(min(8, len(meta_df))).tolist()
blend12_oof, blend12_w, _ = optimize_single_horizon(model_oof, top12_names, target_col=0, score_func=score12_func, max_add=5)
blend12_test = np.tensordot(np.asarray([blend12_w[n] for n in blend12_w]), np.stack([model_test[n][:, 0] for n in blend12_w], axis=0), axes=(0, 0))
for alpha in np.linspace(0.0, 0.45, 10):
    cand_col = np.clip((1.0 - alpha) * core_oof[:, 0] + alpha * blend12_oof, 0.0, 1.0)
    cand = core_oof.copy()
    cand[:, 0] = np.minimum(cand_col, cand[:, 1])
    s = score12_func(cand[:, 0])
    if alpha == 0.0:
        best_alpha12 = 0.0
        best_s12 = s
    elif s > best_s12 + 1e-8:
        best_alpha12 = float(alpha)
        best_s12 = float(s)
core_oof[:, 0] = np.minimum(np.clip((1.0 - best_alpha12) * core_oof[:, 0] + best_alpha12 * blend12_oof, 0.0, 1.0), core_oof[:, 1])
core_test[:, 0] = np.minimum(np.clip((1.0 - best_alpha12) * core_test[:, 0] + best_alpha12 * blend12_test, 0.0, 1.0), core_test[:, 1])
core_oof = enforce_monotonicity(core_oof)
core_test = enforce_monotonicity(core_test)
print("12h refinement alpha:", best_alpha12)

# Per-horizon calibration on OOF
calibrated_oof = core_oof.copy()
calibrated_test = core_test.copy()
calibration_info = {}

for j, h in enumerate(HORIZONS):
    cal_model, info = choose_calibrator(core_oof[:, j], brier_cache[h], seed=SEED + h)
    calibrated_oof[:, j] = cal_model.predict(core_oof[:, j])
    calibrated_test[:, j] = cal_model.predict(core_test[:, j])
    calibration_info[h] = info

calibrated_oof = enforce_monotonicity(np.clip(calibrated_oof, 0.0, 1.0))
calibrated_test = enforce_monotonicity(np.clip(calibrated_test, 0.0, 1.0))

# Auto-check whether pushing 72h toward 1 helps under the local metric
b72_model = brier_from_cache(calibrated_oof[:, 3], brier_cache[72])
ones72 = np.ones(len(calibrated_oof), dtype=float)
b72_one = brier_from_cache(ones72, brier_cache[72])
best72_alpha = 0.0
best72_score = metric_func(calibrated_oof)
for alpha in [0.0, 0.10, 0.20, 0.35, 0.50, 0.70, 0.85, 1.0]:
    cand = calibrated_oof.copy()
    cand[:, 3] = np.clip((1.0 - alpha) * cand[:, 3] + alpha * 1.0, 0.0, 1.0)
    cand = enforce_monotonicity(cand)
    s = metric_func(cand)
    if s > best72_score + 1e-8:
        best72_score = s
        best72_alpha = float(alpha)
if best72_alpha > 0:
    calibrated_oof[:, 3] = np.clip((1.0 - best72_alpha) * calibrated_oof[:, 3] + best72_alpha * 1.0, 0.0, 1.0)
    calibrated_test[:, 3] = np.clip((1.0 - best72_alpha) * calibrated_test[:, 3] + best72_alpha * 1.0, 0.0, 1.0)
    calibrated_oof = enforce_monotonicity(calibrated_oof)
    calibrated_test = enforce_monotonicity(calibrated_test)
print("72h local-better alpha:", best72_alpha, "| model_b72:", round(b72_model, 6), "| one_b72:", round(b72_one, 6))

core_submission = sample_sub.copy()
core_submission[REQUIRED_COLS[1:]] = calibrated_test
validate_submission(core_submission, sample_sub)
core_submission.to_csv("/kaggle/working/submission_core.csv", index=False)

# Conservative optional stage-2 blend with uploaded/public historical submission CSVs
sample_ids = sample_sub["event_id"].tolist()
deny_files = {
    "train.csv", "test.csv", "sample_submission.csv", "metaData.csv",
    "submission.csv", "submission_core.csv", "submission_stage2.csv",
    "submission_external.csv", "submission_breakthrough.csv",
}
candidate_paths = []
for root in SEARCH_ROOTS:
    if not os.path.isdir(root):
        continue
    for path in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        fn = os.path.basename(path)
        if fn in deny_files:
            continue
        if "samplecsv_sub" not in fn.lower() and not re.search(r"sub(?:mission)?[_-]?\d+", fn.lower()):
            continue
        candidate_paths.append(path)

candidate_paths = sorted(set(candidate_paths))
external_frames = []
external_meta = []

for path in candidate_paths:
    cand = safe_read_submission_csv(path, sample_ids)
    if cand is None:
        continue
    arr = cand[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    mad = float(np.mean(np.abs(arr[:, :3] - calibrated_test[:, :3])))
    if mad < 1e-7 or mad > 0.12:
        continue
    score = extract_score_from_name(path, SCORE_LOOKUP)
    external_frames.append(cand)
    external_meta.append({
        "path": path,
        "score": score,
        "sub_idx": extract_submission_index(path),
        "mad_to_core": mad,
        "hash": hash_frame(cand),
    })

dedup_frames = []
dedup_meta = []
seen_hash = set()
for df_c, meta in zip(external_frames, external_meta):
    if meta["hash"] in seen_hash:
        continue
    seen_hash.add(meta["hash"])
    dedup_frames.append(df_c)
    dedup_meta.append(meta)
external_frames = dedup_frames
external_meta = dedup_meta

if external_frames:
    ext_arrays = [df_c[REQUIRED_COLS[1:]].to_numpy(dtype=float) for df_c in external_frames]
    known_scores = [m["score"] for m in external_meta if m["score"] is not None]
    ref_score = max(known_scores) if known_scores else None
    raw_w = []
    for arr, meta in zip(ext_arrays, external_meta):
        if ref_score is None or meta["score"] is None:
            score_w = 0.85
        else:
            score_w = float(np.exp((meta["score"] - ref_score) / 0.0025))
        div = float(np.mean(np.abs(arr[:, :3] - calibrated_test[:, :3])))
        div_w = float(np.clip(div / 0.010, 0.60, 1.30))
        raw_w.append(score_w * div_w)
    raw_w = np.asarray(raw_w, dtype=float)
    if raw_w.sum() <= 0:
        raw_w = np.ones_like(raw_w)
    raw_w = raw_w / raw_w.sum()

    ext_mean = np.tensordot(raw_w, np.stack(ext_arrays, axis=0), axes=(0, 0))
    ext_rank = np.zeros_like(ext_mean)
    for j in range(4):
        rank_stack = np.stack([pct_rank(arr[:, j]) for arr in ext_arrays], axis=0)
        rank_signal = np.tensordot(raw_w, rank_stack, axes=(0, 0))
        ext_rank[:, j] = map_rank_to_distribution(calibrated_test[:, j], rank_signal)
    ext_consensus = enforce_monotonicity(0.72 * ext_mean + 0.28 * ext_rank)

    sel_stack_test = np.stack([model_test[n] for n in sel_names], axis=0)
    uncertainty = np.std(sel_stack_test[:, :, :3], axis=0)
    scale24 = np.quantile(uncertainty[:, 0], 0.80) + 1e-8
    scale48 = np.quantile(uncertainty[:, 1], 0.80) + 1e-8
    scale72 = np.quantile(uncertainty[:, 2], 0.80) + 1e-8
    unc_norm = np.column_stack([
        np.clip(uncertainty[:, 0] / scale24, 0.65, 1.35),
        np.clip(uncertainty[:, 1] / scale48, 0.65, 1.35),
        np.clip(uncertainty[:, 2] / scale72, 0.65, 1.35),
        np.ones(len(test), dtype=float),
    ])

    strength = min(1.0, 0.25 + 0.12 * len(ext_arrays))
    alpha_base = np.array([0.04, 0.09, 0.13, 0.00 if best72_alpha > 0.5 else 0.03], dtype=float)
    alpha = alpha_base[None, :] * strength * unc_norm

    final_test = (1.0 - alpha) * calibrated_test + alpha * ext_consensus
    final_test = enforce_monotonicity(np.clip(final_test, 0.0, 1.0))

    external_summary = pd.DataFrame(external_meta)
    external_summary["weight"] = raw_w
    external_summary = external_summary.sort_values("weight", ascending=False)
    external_summary.to_csv("/kaggle/working/external_blend_summary.csv", index=False)

    submission = sample_sub.copy()
    submission[REQUIRED_COLS[1:]] = final_test
    validate_submission(submission, sample_sub)
    submission.to_csv("/kaggle/working/submission_external.csv", index=False)
else:
    submission = core_submission.copy()
    external_summary = pd.DataFrame()
    submission.to_csv("/kaggle/working/submission_external.csv", index=False)

submission.to_csv("/kaggle/working/submission.csv", index=False)
validate_submission(submission, sample_sub)

oof_export = pd.DataFrame({
    "event_id": train["event_id"],
    "event": train["event"],
    "time_to_hit_hours": train["time_to_hit_hours"],
    "oof_12h": calibrated_oof[:, 0],
    "oof_24h": calibrated_oof[:, 1],
    "oof_48h": calibrated_oof[:, 2],
    "oof_72h": calibrated_oof[:, 3],
})
oof_export.to_csv("/kaggle/working/oof_predictions.csv", index=False)

summary = {
    "best_single_model": meta_df.iloc[0].to_dict() if len(meta_df) else None,
    "candidate_names": candidate_names,
    "core_weights": {k: float(v) for k, v in core_weights.items()},
    "core_hybrid_oof": float(metric_func(core_oof)),
    "rank_alpha": float(best_rank_alpha),
    "alpha12": float(best_alpha12),
    "alpha72": float(best72_alpha),
    "calibration": calibration_info,
    "used_external_csvs": int(len(external_frames) if external_frames else 0),
}
with open("/kaggle/working/ensemble_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:")
print("/kaggle/working/model_scores.csv")
print("/kaggle/working/oof_predictions.csv")
print("/kaggle/working/submission_core.csv")
print("/kaggle/working/submission_external.csv")
print("/kaggle/working/submission.csv")


HAS_XGB: True | HAS_LGB: True | HAS_CAT: True
Core dataset: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
Base models scheduled: 22

===== aft__normal_d3_s08 (aft) =====
Hybrid=0.835425 | B12=0.106910 B24=0.144847 B48=0.137549 B72=0.352526

===== aft__normal_d4_s12 (aft) =====
Hybrid=0.833227 | B12=0.106674 B24=0.149575 B48=0.140463 B72=0.356389

===== aft__logistic_d3_s10 (aft) =====
Hybrid=0.813856 | B12=0.114856 B24=0.163870 B48=0.161880 B72=0.407912

===== aft__logistic_d2_s16 (aft) =====
Hybrid=0.812381 | B12=0.111038 B24=0.162030 B48=0.163992 B72=0.413485

===== aft__extreme_d3_s10 (aft) =====
Hybrid=0.812036 | B12=0.101555 B24=0.158616 B48=0.163970 B72=0.416207

===== cox__d3_lr035 (cox) =====
Hybrid=0.964559 | B12=0.068587 B24=0.036952 B48=0.024666 B72=0.002101

===== cox__d4_lr030 (cox) =====
Hybrid=0.963271 | B12=0.068647 B24=0.038165 B48=0.025990 B72=0.003481

===== cox__d2_lr050 (cox) =====
Hybrid=0.965821 | B12=0.066604 B24=0.035558 B48=0.023475 B72=0.000904

===